<pre>
LangChain Core
│
├── Interface: VectorStore (uniform)
│
├── langchain-milvus      (infra DB)
├── langchain-pinecone   (managed DB)
├── langchain-qdrant     (infra DB)
│
└── langchain-community
    ├── FAISS            (library)
    ├── Supabase         (pgvector)
    └── Weaviate         (fast-moving)
</pre>


In [1]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma 
import pandas as pd

In [2]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [3]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [4]:
## current working directory
import os
current_dir = os.getcwd()
current_dir  

'/Users/ioi/Documents/RAG_Course/Pro_Package/src/Nit_langchain/vectorStore'

In [5]:
## code for persisting the documents in Chroma vector store
## Pricing --- https://platform.openai.com/docs/pricing#embeddings
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory='my_chroma_db',
    collection_name='sample'
)
####============================== INMEMORY STORAGE =============================####

# code for inmeomry (RAM storage) vector store using Chroma
# vector_store = Chroma(
#     embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
#     collection_name='sample'
# )

In [6]:
# add documents
vector_store.add_documents(docs) 

['e1b38815-11e5-4fc6-8dc8-6d551e9ba655',
 'f414088a-6e85-41d1-9a56-d16bb6983b44',
 '218f1c00-3d25-4928-8027-4d09706d3787',
 '497e3176-587a-4898-8bb4-83240a52701b',
 'd7467937-7b46-4036-9a55-e8f4231a5ade']

```python
{
    "ids": List[str],                    # ALWAYS present
    "documents": List[str] | None,
    "metadatas": List[dict] | None,
    "embeddings": np.ndarray | None      # shape = (n, embedding_dim)
}

In [10]:
# retrieve documents with embeddings and metadata
data = vector_store.get(
    include=["documents", "metadatas", "embeddings"]
)
# type(data)--> return type -->dict
# data.keys()--> dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
# type(data["ids"]) --> list
# type(data["documents"]) --> list of original text as per docs
# data["metadatas"] --> list of metadata dicts
# type(data["embeddings"])--> 2-D numpy.ndarray
# data["embeddings"].shape
# Convert embeddings to DataFrame for better visualization

df = pd.DataFrame({
    "id": data["ids"],
    "document": data["documents"],
    "metadata": data["metadatas"],
    "embedding": pd.Series(list(data["embeddings"]))  # 🔑 KEY FIX
})

df.head() 

,id,document,metadata,embedding
0,e1b38815-11e5-4fc6-8dc8-6d551e9ba655,Virat Kohli is one of the most successful and ...,{'team': 'Royal Challengers Bangalore'},"[0.014453452080488205, 0.04768107086420059, 0...."
1,f414088a-6e85-41d1-9a56-d16bb6983b44,Rohit Sharma is the most successful captain in...,{'team': 'Mumbai Indians'},"[0.04187608137726784, -0.022892257198691368, 0..."
2,218f1c00-3d25-4928-8027-4d09706d3787,"MS Dhoni, famously known as Captain Cool, has ...",{'team': 'Chennai Super Kings'},"[0.09531600773334503, -0.024628914892673492, 0..."
3,497e3176-587a-4898-8bb4-83240a52701b,Jasprit Bumrah is considered one of the best f...,{'team': 'Mumbai Indians'},"[-0.034180302172899246, 0.049170952290296555, ..."
4,d7467937-7b46-4036-9a55-e8f4231a5ade,Ravindra Jadeja is a dynamic all-rounder who c...,{'team': 'Chennai Super Kings'},"[0.008311678655445576, -0.003894301364198327, ..."


In [12]:
df.loc[0, 'embedding']  # embedding vector for first document

array([0.01445345, 0.04768107, 0.03671075, ..., 0.00683347, 0.00905001,
       0.01097032], shape=(1536,))

In [13]:
# search documents, following will return top 2 similar documents of type langchain_core.documents.Document
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
) 

[Document(id='497e3176-587a-4898-8bb4-83240a52701b', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='d7467937-7b46-4036-9a55-e8f4231a5ade', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [14]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='497e3176-587a-4898-8bb4-83240a52701b', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.0509319305419922),
 (Document(id='d7467937-7b46-4036-9a55-e8f4231a5ade', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.2550573348999023)]

In [15]:
# meta-data filtering
## How many players are from Chennai Super Kings?
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='d7467937-7b46-4036-9a55-e8f4231a5ade', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8200222253799438),
 (Document(id='218f1c00-3d25-4928-8027-4d09706d3787', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8259756565093994)]

In [16]:
df['id'] 

0    e1b38815-11e5-4fc6-8dc8-6d551e9ba655
1    f414088a-6e85-41d1-9a56-d16bb6983b44
2    218f1c00-3d25-4928-8027-4d09706d3787
3    497e3176-587a-4898-8bb4-83240a52701b
4    d7467937-7b46-4036-9a55-e8f4231a5ade
Name: id, dtype: object

In [17]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='e1b38815-11e5-4fc6-8dc8-6d551e9ba655', document=updated_doc1)


In [18]:
# view documents
data_update=vector_store.get(include=['embeddings','documents', 'metadatas'])
## print as DataFrame
df_update = pd.DataFrame({
    "id": data_update["ids"],
    "document": data_update["documents"],
    "metadata": data_update["metadatas"],
    "embedding": pd.Series(list(data_update["embeddings"]))  # 🔑 KEY FIX
})  
df_update.head()

,id,document,metadata,embedding
0,e1b38815-11e5-4fc6-8dc8-6d551e9ba655,"Virat Kohli, the former captain of Royal Chall...",{'team': 'Royal Challengers Bangalore'},"[0.03273560479283333, 0.05454500392079353, 0.0..."
1,f414088a-6e85-41d1-9a56-d16bb6983b44,Rohit Sharma is the most successful captain in...,{'team': 'Mumbai Indians'},"[0.04187608137726784, -0.022892257198691368, 0..."
2,218f1c00-3d25-4928-8027-4d09706d3787,"MS Dhoni, famously known as Captain Cool, has ...",{'team': 'Chennai Super Kings'},"[0.09531600773334503, -0.024628914892673492, 0..."
3,497e3176-587a-4898-8bb4-83240a52701b,Jasprit Bumrah is considered one of the best f...,{'team': 'Mumbai Indians'},"[-0.034180302172899246, 0.049170952290296555, ..."
4,d7467937-7b46-4036-9a55-e8f4231a5ade,Ravindra Jadeja is a dynamic all-rounder who c...,{'team': 'Chennai Super Kings'},"[0.008311678655445576, -0.003894301364198327, ..."


In [19]:
# delete document
vector_store.delete(ids=['e1b38815-11e5-4fc6-8dc8-6d551e9ba655'])
# view documents after deletion as DataFrame
data_after_delete=vector_store.get(include=['embeddings','documents', 'metadatas'])
df_after_delete = pd.DataFrame({
    "id": data_after_delete["ids"],
    "document": data_after_delete["documents"],
    "metadata": data_after_delete["metadatas"],
    "embedding": pd.Series(list(data_after_delete["embeddings"]))  # 🔑 KEY FIX
})  
df_after_delete.head()

,id,document,metadata,embedding
0,f414088a-6e85-41d1-9a56-d16bb6983b44,Rohit Sharma is the most successful captain in...,{'team': 'Mumbai Indians'},"[0.04187608137726784, -0.022892257198691368, 0..."
1,218f1c00-3d25-4928-8027-4d09706d3787,"MS Dhoni, famously known as Captain Cool, has ...",{'team': 'Chennai Super Kings'},"[0.09531600773334503, -0.024628914892673492, 0..."
2,497e3176-587a-4898-8bb4-83240a52701b,Jasprit Bumrah is considered one of the best f...,{'team': 'Mumbai Indians'},"[-0.034180302172899246, 0.049170952290296555, ..."
3,d7467937-7b46-4036-9a55-e8f4231a5ade,Ravindra Jadeja is a dynamic all-rounder who c...,{'team': 'Chennai Super Kings'},"[0.008311678655445576, -0.003894301364198327, ..."
